In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from utils.data_utils import data_preprocessing
from utils.data_utils import evaluate_rf
import utils.analysis_functions as af


TEST_SIZE = 0.2
RANDOM_KEY = 21

df = pd.read_parquet("data/dataset.parquet")
npp_df=pd.read_csv('data/processed/PID_npp.csv')
csc_df= pd.read_csv('data/processed/PID_csc_upsampled.csv')

PID_df, csc_df, npp_df = af.cleaning(df, csc_df, npp_df)

npp_df=pd.read_csv('data/processed/npp_heterosced.csv')



In [2]:
PID_df.head(2)

,PID,Raos_Q,Functional_Evenness,Functional_Dispersion,Functional_Divergences,Mean Pairwise D,CHELSA_BIO_Annual_Mean_Temperature,CHELSA_BIO_Annual_Precipitation,CHELSA_BIO_Precipitation_Seasonality,CrowtherLab_SoilMoisture_intraAnnualSD_downsampled10km,...,EarthEnvTopoMed_Northness,BHAGE,managed,ownership,biome,Species Richness,Shannon Diversity,Simpson's Index,Shannon Equitabiltiy Index,source_file
0,1_26_109_100_301,1.081759,0.688448,2.626585,0.523358,4.069748,59.533163,796.36676,35.801643,18.797384,...,0.2692,NaN,0.0,other,Temperate broadleaf forests,6,0.860597,0.384222,0.480308,dataset45.csv
1,1_26_109_100_301,1.081759,0.688448,2.626585,0.523358,4.069748,59.533163,796.36676,35.801643,18.797384,...,0.2692,NaN,0.0,other,Temperate broadleaf forests,6,0.860597,0.384222,0.480308,dataset45.csv


In [ ]:
csc_df['csc'] = (csc_df['csc']-min(csc_df['csc']))/(max(csc_df['csc'])-min(csc_df['csc']))
csc_df["BHAGE"] = csc_df["BHAGE"].fillna("0.0")
csc_df=csc_df[['PID', 'csc']]

y=['transformed npp', 'csc'] 

sd_df, fd_df=data_preprocessing(df, npp_df, csc_df)

fd_df= fd_df.sample(n=10000, random_state=RANDOM_KEY)
sd_df= sd_df.sample(n=10000, random_state=RANDOM_KEY)


# fd_x=fd_df.drop(columns=y+['PID'])
# fy = np.column_stack([fd_df['transformed npp'], fd_df['csc']])

# sd_x = sd_df.drop(columns=y + ['PID'])
# sy = np.column_stack([sd_df['transformed npp'], sd_df['csc']])

In [ ]:
fd_df.to_csv( "data/sample/fd_df.csv", index=False)
sd_df.to_csv( "data/sample/sd_df.csv", index=False)

In [7]:
RABDOM_KEY = 5

fy = pd.DataFrame(fy, columns=['transformed npp', 'csc'])
sy= pd.DataFrame(sy, columns=['transformed npp', 'csc'])

sub_fd_x= fd_x.sample(n=10000, random_state=RANDOM_KEY)
sub_sd_x= sd_x.sample(n=10000, random_state=RANDOM_KEY)
sub_fy= fy.sample(n=10000, random_state=RANDOM_KEY)
sub_sy= sy.sample(n=10000, random_state=RANDOM_KEY)

In [9]:
sub_fd_x.to_csv('data/sample/functional_div_x.csv', index=False)
sub_sd_x.to_csv('data/sample/species_div_x.csv', index=False)

sub_fy.to_csv('data/sample/functional_div_y.csv', index=False)
sub_sy.to_csv('data/sample/species_div_y.csv', index=False)

In [ ]:
# Temporal Conditional Heteroscedasticity

# autocorr_df = (
#     npp_df.groupby("PID")["Npp"]
#     .apply(af.arch_coeff)
#     .reset_index(name="transformed npp")
# )
# npp_df= autocorr_df[["PID", "transformed npp"]]



fX_train, fX_test, fy_train, fy_test = train_test_split(fd_x, fy, test_size=TEST_SIZE, random_state=RANDOM_KEY, stratify=fy)
sX_train, sX_test, sy_train, sy_test = train_test_split(sd_x, sy, test_size=TEST_SIZE, random_state=RANDOM_KEY, stratify=sy)

fd_reg = RandomForestRegressor(random_state=RANDOM_KEY, max_depth=3)
fd_reg.fit(fX_train, fy_train)


sd_reg = RandomForestRegressor(random_state=RANDOM_KEY, max_depth=3)
sd_reg.fit(sX_train, sy_train)


evaluate_rf(fX_test, fy_test, fd_reg, feature_names=fd_x.columns, importance= True, div_type= 'f')
evaluate_rf(sX_test, sy_test, sd_reg, feature_names=sd_x.columns, importance= True,  div_type= 's')